In [6]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from dataclasses import dataclass
import json

@dataclass
class DataBundle:
    dates: pd.DatetimeIndex
    instruments: list[str]
    speeds: list[str]
    signals: np.ndarray    # (T, N_inst, N_speeds)
    returns: np.ndarray    # (T, N_inst)
    risk_free: np.ndarray  # (T,)

def load_data(prices_path, signals_path, cash_path) -> DataBundle:
    prices = pd.read_csv(prices_path, parse_dates=["date"]).set_index("date").sort_index()
    signals = pd.read_csv(signals_path, parse_dates=["date"]).set_index("date").sort_index()
    cash = pd.read_csv(cash_path, parse_dates=["date"]).set_index("date").sort_index()

    instruments = list(prices.columns)
    first_inst = instruments[0]
    speeds = [c.replace(f"{first_inst}_", "") for c in signals.columns if c.startswith(f"{first_inst}_")]
    
    # Forward returns for PnL calculation
    fwd_returns = prices.pct_change().shift(-1)
    
    # Daily Risk Free Rate from 3mo treasury
    rf_daily = (cash['3mo'].reindex(prices.index).ffill().fillna(0) / 100) / 252
    
    common_idx = prices.index.intersection(signals.index).dropna()
    sig_arr = np.zeros((len(common_idx), len(instruments), len(speeds)))
    
    for j, inst in enumerate(instruments):
        for k, spd in enumerate(speeds):
            col = f"{inst}_{spd}"
            sig_arr[:, j, k] = signals.loc[common_idx, col].values
            
    ret_arr = fwd_returns.loc[common_idx].values
    rf_arr = rf_daily.loc[common_idx].values

    return DataBundle(dates=common_idx, instruments=instruments, speeds=speeds,
                      signals=sig_arr, returns=ret_arr, risk_free=rf_arr)

In [7]:
class PortfolioEngine:
    def __init__(self, data: DataBundle, vol_floor=0.12): 
        self.data = data
        self.vol_floor = vol_floor
        
        # Mean across trend horizons (T, N)
        self._forecast = np.nanmean(data.signals, axis=2)
        
        # Calculate Rolling Volatility (63-day) with Floor
        self._vol = self._calculate_vol(data.returns)
        
        # Risk-normalized returns: prevents low-vol assets from over-leveraging
        self._norm_returns = data.returns / self._vol

    def _calculate_vol(self, returns, window=63):
        T, N = returns.shape
        vol = np.zeros_like(returns)
        for t in range(window, T):
            vol[t] = np.nanstd(returns[t-window:t], axis=0) * np.sqrt(252)
        vol[:window] = np.nanstd(returns[:window], axis=0) * np.sqrt(252)
        return np.maximum(vol, self.vol_floor)

    def weights_to_pnl(self, weights):
        daily_pnl = np.nansum(self._forecast * weights * self._norm_returns, axis=1)
        return daily_pnl - self.data.risk_free

    def sortino_ratio(self, weights):
        pnl = self.weights_to_pnl(weights)
        mean_ret = np.nanmean(pnl) * 252
        downside_diff = np.where(pnl < 0, pnl, 0)
        downside_vol = np.std(downside_diff) * np.sqrt(252)
        return mean_ret / downside_vol if downside_vol > 1e-9 else 0.0

In [8]:
class RobustOptimiser:
    def __init__(self, data: DataBundle, diversification_strength=1.0):
        self.data = data
        self.div_strength = diversification_strength # Higher = More equal weights
        self.n = len(data.instruments)
        
        # Calculate daily 'Risk-Adjusted' PnL per instrument
        # (Signal * Return) / Volatility
        # Note: We shift signals by 1 to represent trading at t based on t-1 signals
        self.unit_pnls = (self.data.signals.shift(1) * self.data.returns / self.data.vol).fillna(0)

    def objective(self, weights):
        # 1. Performance Metric (Annualized Sharpe)
        portfolio_pnl = (self.unit_pnls * weights).sum(axis=1)
        mu, sigma = portfolio_pnl.mean() * 252, portfolio_pnl.std() * np.sqrt(252)
        sharpe = mu / sigma if sigma > 1e-9 else 0
        
        # 2. Diversification Penalty (L2 Regularization)
        # This penalizes weights that deviate from 1/N (10% each)
        target_weight = 1.0 / self.n
        penalty = self.div_strength * np.sum((weights - target_weight)**2)
        
        # We maximize (Sharpe - Penalty)
        return -(sharpe - penalty)

    def solve(self, min_w=0.05, max_w=0.15):
        # IMPROVEMENT 3: Tightened Bounds (e.g., max 15% per instrument)
        # This is a hard-cap on polarization.
        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0})
        bounds = [(min_w, max_w) for _ in range(self.n)]
        
        initial_guess = np.ones(self.n) / self.n
        
        res = minimize(self.objective, initial_guess, 
                       method='SLSQP', bounds=bounds, constraints=constraints)
        
        return res.x, -res.fun

In [9]:
class Optimiser:
    def __init__(self, engine: PortfolioEngine, max_weight=0.15): 
        self.engine = engine
        self.n = len(engine.data.instruments)
        self.max_weight = max_weight

    def objective(self, weights):
        # 1. Performance (Sortino)
        sortino = self.engine.sortino_ratio(weights)
        
        # 2. Diversification Penalty (L2) - Pulls weights toward 10%
        target = 1.0 / self.n
        div_penalty = 0.6 * np.sum((weights - target)**2)
        
        # 3. Boundary Penalty - Hard enforcement of the ceiling
        bound_penalty = np.sum(np.maximum(0, weights - self.max_weight)**2) * 100
        
        return -(sortino - div_penalty - bound_penalty)

    def fit(self):
        # Start with Inverse Volatility weights (Risk Parity)
        avg_vols = np.mean(self.engine._vol, axis=0)
        inv_vol_weights = (1/avg_vols) / np.sum(1/avg_vols)
        
        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0})
        # Strict Bounds: 3% minimum, 15% maximum per instrument
        bounds = [(0.03, self.max_weight) for _ in range(self.n)] 
        
        res = minimize(self.objective, inv_vol_weights, 
                       method='SLSQP', bounds=bounds, constraints=constraints)
        
        return res.x, self.engine.sortino_ratio(res.x)

In [13]:
# ═══════════════════════════════════════════════════════════════════════════
# EXECUTION
# ═══════════════════════════════════════════════════════════════════════════

# 1. Load Data (Adjust paths if files are in a specific folder)
data = load_data("data_3/prices.csv", "data_3/signals.csv", "data_3/cash_rate.csv")

# 2. Run Optimization (12% Vol Floor, 15% Weight Ceiling)
engine = PortfolioEngine(data, vol_floor=0.12) 
optimiser = Optimiser(engine, max_weight=0.15)
best_weights, best_metric = optimiser.fit()

# 3. Print Final Coefficients
print("\n" + "="*45)
print(f"{'INSTRUMENT':<15} | {'OPTIMAL WEIGHT':<15}")
print("="*45)
for inst, w in zip(data.instruments, best_weights):
    bar = "█" * int(w * 100)
    print(f"{inst:<15} | {w:>14.2%}  {bar}")
print("-" * 45)
print(f"Portfolio Sortino Ratio: {best_metric:.3f}")

# 4. Export to CSV
pd.DataFrame({"instrument": data.instruments, "coefficient": best_weights}).to_csv("final_coefficients.csv", index=False)


INSTRUMENT      | OPTIMAL WEIGHT 
INSTRUMENT_1    |          3.00%  ███
INSTRUMENT_2    |          4.96%  ████
INSTRUMENT_3    |          3.00%  ███
INSTRUMENT_4    |          3.00%  ███
INSTRUMENT_5    |         15.00%  ███████████████
INSTRUMENT_6    |         13.28%  █████████████
INSTRUMENT_7    |         12.76%  ████████████
INSTRUMENT_8    |         15.00%  ███████████████
INSTRUMENT_9    |         15.00%  ███████████████
INSTRUMENT_10   |         15.00%  ███████████████
---------------------------------------------
Portfolio Sortino Ratio: 0.043


/var/folders/wj/n3s45m710_xghx8ld878v_qh0000gn/T/ipykernel_1522/2567496590.py:7: RuntimeWarning: Mean of empty slice
  self._forecast = np.nanmean(data.signals, axis=2)
